<div class="alert alert-success">  
</div>

# <font size='6' color='darkred'>Gaussian Naive Bayes (GaussianNB)</font>

## <font size='4' color='darkcyan'>Competition Kaggle (Synthetic Datasets)</font>
<div class="alert alert-success">  
    <h1 align="center" style="color:darkgreen;">Binary Classification of Machine Failures</h1>  
    <h3 align="center" style="color:darkgreen;">Playground Series - Season 3, Episode 17</h3>    
</div>


In [ ]:
import warnings # suppress warnings
warnings.filterwarnings('ignore')
#########################################
import os
import gc
import glob
import random
import numpy as np 
import pandas as pd
import seaborn as sns
from tqdm import tqdm
from scipy import stats
from pathlib import Path
#########################################
import matplotlib.pyplot as plt
import plotly.figure_factory as ff
import plotly.express as px
%matplotlib inline
!ls ../input/*
#########################################
pd.set_option('display.max_columns', 500)

## <span style="color:darkcyan;">Exploratory Data Analysis</span>

<div class="alert alert-success">  
</div>

In [ ]:
df_train  = pd.read_csv('/kaggle/input/playground-series-s3e17/train.csv', index_col='id')
df_test   = pd.read_csv('/kaggle/input/playground-series-s3e17/test.csv', index_col='id')
df_sample = pd.read_csv('/kaggle/input/playground-series-s3e17/sample_submission.csv')

# display(df_train , df_test, df_sample)
display(df_train.shape , df_test.shape, df_sample.shape)

In [ ]:
display(df_train.columns.tolist())

In [ ]:
# Original data
df_org = pd.read_csv('/kaggle/input/machine-failure-predictions/machine failure.csv')

pd.DataFrame(data= {'Number': df_org['Machine failure'].value_counts(), 
                    'Percent': df_org['Machine failure'].value_counts(normalize=True)})

In [ ]:
display(df_train.info())
# df_train.describe().transpose()

In [ ]:
display(df_train.nunique())

### <span style="color:darkred;">Check Null Values</span>

In [ ]:
MV1 = df_train.isnull().sum()
MV2 = df_test.isnull().sum()

print(':' * 25)
print(f'Missing Value df_train:\n{MV1[MV1 > 0]}')
print(':' * 25)
print(f'Missing Value df_test:\n{MV2[MV2 > 0]}')
print(':' * 25)

## <span style="color:darkred;">Train | Test | Target</span>
#### Let's see how unbalanced the dataset is:

In [ ]:
train  = df_train.copy()
test   = df_test.copy()
target = train.pop('Machine failure')

target.value_counts().plot(kind='barh', figsize=(12,1), title='Target Count', color=['darkcyan','red'])
plt.gca().set_facecolor('lightgray')

pd.DataFrame(data= {'Number': target.value_counts(), 'Percent': target.value_counts(normalize=True)})

## <span style="color:darkred;">Features</span>

In [ ]:
features = train.columns.tolist()

# Categorical features
cat_features = ['Product ID', 'Type']

# Binary features
bin_features = ['TWF', 'HDF', 'PWF', 'OSF', 'RNF']

# Numerical features
num_features = [f for f in features if f not in (cat_features + bin_features)]
              
print(':' * 40)
print('The number of Categorical_features:', len(cat_features)) 
print('The number of Binary_features:', len(bin_features))  
print('The number of Numerical_features:', len(num_features)) 
print(':' * 40)
print('The total number of features:', len(features))
print(':' * 40)

## <span style="color:darkred;">Categorical features</span>

In [ ]:
for f in cat_features:
    
    print('\t' , f)
    n_f = train[f].value_counts()
    p_f = train[f].value_counts(normalize=True)
    display(pd.DataFrame(data= {'Number': n_f, 'Percent': p_f}))

## <span style="color:darkred;">Binary features</span>

In [ ]:
dfv = pd.DataFrame(data= {'Value': ['Number 0', 'Percent 0', '', 'Number 1', 'Percent 1']})

for f in bin_features: 
    n_f = train[f].value_counts()
    p_f = train[f].value_counts(normalize=True)
    dfv[f] = [n_f[0], p_f[0], '', n_f[1], p_f[1]]
    
dfv.set_index('Value')

## <span style="color:darkred;">Histograms of the Numerical features</span>

In [ ]:
sns.set()
plt.style.use('seaborn-whitegrid') 
_, axs = plt.subplots(2, 3, figsize=(15,10), facecolor='lightgray')

for f, ax in zip(num_features, axs.ravel()):
    ax.set_facecolor('lightyellow')
    ax.hist(train[f], bins=100, color='red')
    ax.set_title(f'Feature: {f}', fontsize=10)

plt.suptitle('Histograms of the Numerical Features (5 Columns)', y=0.94, fontsize=16)
plt.show()

## <span style="color:darkred;">Correlation Matrix</span>

In [ ]:
cor_matrix = df_train[num_features + bin_features  + ['Machine failure']].corr().round(2)
fig = plt.figure(figsize=(8,8));
# mask = np.zeros_like(cor_matrix)
# mask[np.triu_indices_from(mask)] = True
plt.gca().set_facecolor('lightyellow')

cmap=sns.diverging_palette(240, 10, s=75, l=50, sep=1, n=6, center='light', as_cmap=False);
sns.heatmap(cor_matrix, center=0, annot=False, cmap=cmap, mask=False, linewidths=2);
plt.show()

In [ ]:
corr = df_train[num_features + bin_features  + ['Machine failure']].corr(numeric_only=True)
corr.style.background_gradient(cmap='Blues')

## <span style="color:darkred;">Evaluation Metric (AUC)</span>

 In this Kaggle challenge; submissions are evaluated on area under the ROC curve between the predicted probability and the observed target.

In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve, auc

def roc_auc(true_list, pred_list, a, b):
    
    fpr, tpr, _ = roc_curve(true_list, pred_list)    
    roc_auc = auc(fpr, tpr)

    # print(f'FPR: {fpr}')
    # print(f'TPR: {tpr}')
    # print(f'{list(zip(fpr,tpr))}') 
    print(f'\n>>>>> ROC_AUC: %0.6f <<<<<\n' %roc_auc)
    
    sns.set()
    plt.style.use('seaborn-whitegrid')
    plt.figure(figsize=(a, b), facecolor='lightgray')
    plt.gca().set_facecolor('lightyellow')
    plt.plot(fpr, tpr, color='darkorange', lw=2, label='ROC curve')
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    plt.xlim([-0.01, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('The area under the ROC curve\n')
    plt.legend(loc="lower right")
    plt.show()

<div class="alert alert-success">  
</div>

## <span style="color:darkcyan;">Data Preprocessing</span>

<div class="alert alert-success">  
</div>

In [ ]:
from sklearn.preprocessing import StandardScaler, LabelEncoder

In [ ]:
df_org.drop('UDI', axis=1, inplace=True)
df_train = pd.concat([df_train, df_org], axis = 0).reset_index(drop=True)
df_train

### <span style="color:darkred;">Let's scale the numerical features</span>

In [ ]:
scaler = StandardScaler()

df_train[num_features] = scaler.fit_transform(df_train[num_features])
df_test[num_features] = scaler.fit_transform(df_test[num_features])

### <span style="color:darkred;">Columns Rename</span>

XGBoost doesn't work with column names that have "[" or "]" in them.

In [ ]:
def fix_names(df):
    df.columns = df.columns.str.replace('[\[\]]', '', regex=True)
    return df

df_train = fix_names(df_train)
df_test  = fix_names(df_test)

### <span style="color:darkred;">Dropping the feature: 'Product ID'</span>

This feature has 9976 different shapes and does not help in calculations.

In [ ]:
display(pd.DataFrame(data= {'Number': df_train['Product ID'].value_counts(), 'Percent': df_train['Product ID'].value_counts(normalize=True)}))

In [ ]:
df_train = df_train.drop(['Product ID'] , axis=1)
df_test = df_test.drop(['Product ID'] , axis=1)

### <span style="color:darkred;">Convert Categorical features</span>

The first method: pandas.get_dummies (OneHotEncoder)

In [ ]:
features = df_train.columns.tolist()
features.remove('Machine failure')

# Categorical features
cat_features = ['Type']

# Binary features
bin_features = ['TWF', 'HDF', 'PWF', 'OSF', 'RNF']

# Numerical features
num_features = [f for f in features if f not in (cat_features + bin_features)]

In [ ]:
train_code = pd.get_dummies(df_train, columns=cat_features)
test_code = pd.get_dummies(df_test, columns=cat_features)
target = train_code.pop('Machine failure')

train_code.shape, test_code.shape, target.shape

<div class="alert alert-success">  
</div>

## <span style="color:darkcyan;">:::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::</span>

## <span style="color:darkred;">Gaussian Naive Bayes (GaussianNB)</span>

## <span style="color:darkcyan;">:::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::</span>

In [ ]:
from sklearn.naive_bayes import GaussianNB

from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import QuantileTransformer

In [ ]:
transformed = pd.DataFrame(QuantileTransformer(output_distribution='normal').fit_transform(train_code))

pipeline = make_pipeline(QuantileTransformer(output_distribution='normal'), GaussianNB())
pipeline.fit(train_code, target)

In [ ]:
cross_val_score(pipeline, train_code, target, scoring='roc_auc', cv=7).mean()

In [ ]:
roc_auc(target, pipeline.predict_proba(train_code)[:,1], 5, 5)

In [ ]:
preds_bayes = pipeline.predict_proba(test_code)[:,1]
preds_bayes

## <span style="color:darkred;">Prediction Histogram</span>

In [ ]:
sns.set()
plt.hist(preds_bayes, bins=50)
plt.gca().set_facecolor('lightblue')
min(preds_bayes), max(preds_bayes)

<div class="alert alert-success">  
</div>

## <span style="color:darkred;">Submission - BAYES </span>

In [ ]:
sub_bayes = df_sample.copy()
sub_bayes['Machine failure'] = preds_bayes
sub_bayes.to_csv('submission_bayes.csv',index=False)
!ls

## <span style="color:darkred;">Ensembling & Submission</span>

Thanks to: **@tetsutani**  
https://www.kaggle.com/code/tetsutani/ps3e17-eda-ensemble-ml-pipeline-shap

Thanks to: **@jimmyyeung**  
https://www.kaggle.com/code/jimmyyeung/ps3-17-machine-failure-catboost-top-21

In [ ]:
sub_import1 = pd.read_csv('/kaggle/input/ps3e17s97778/submission.csv')
sub_import2 = pd.read_csv('/kaggle/input/ps3e17s97788/submission.csv')

In [ ]:
sub = df_sample.copy()
sub['Machine failure'] = (0.05 * sub_bayes['Machine failure']) + (0.05 * sub_import1['Machine failure']) + (0.90 * sub_import2['Machine failure'])
sub.to_csv('submission.csv',index=False)
!ls

## <span style="color:darkred;">Ensembling Histograms </span>

In [ ]:
hist_data = [sub.iloc[:, 1], sub_import2.iloc[:, 1], sub_import1.iloc[:, 1], sub_bayes.iloc[:, 1]] 
group_labels = ['Submission', 'Submission_import2', 'Submission_import1', 'Submission_bayes']
    
fig = ff.create_distplot(hist_data, group_labels, bin_size=.2, show_hist=False, show_rug=False)
fig.show() 

<div class="alert alert-success">  
</div>

<div class="alert alert-success">  
</div>